In [1]:
import os
import sys
import requests
import xmltodict
import pymysql
from datetime import datetime

# 환경변수
API_URL = os.getenv("API_URL", "https://data.ex.co.kr/openapi/safeDriving/forecast")
API_KEY = os.getenv("API_KEY")                     # 발급받은 인증키 (환경변수)
DB_HOST = os.getenv("DB_HOST", "localhost")
DB_PORT = int(os.getenv("DB_PORT", "3306"))
DB_USER = os.getenv("DB_USER", "root")
DB_PASS = os.getenv("DB_PASSWORD")
DB_NAME = os.getenv("DB_NAME_TRAFFIC") or os.getenv("DB_NAME") or ""

# 필수 환경변수 체크
if not API_KEY:
    sys.exit("환경변수 API_KEY가 설정되어 있지 않습니다. API 키를 설정하세요.")
if not DB_NAME:
    sys.exit("환경변수 DB_NAME_TRAFFIC (또는 DB_NAME)이 설정되어 있지 않습니다. DB 이름을 설정하세요.")
if DB_PASS == "":
    sys.exit("환경변수 DB_PASS (또는 DB_PASSWORD)가 설정되어 있지 않습니다. MySQL 비밀번호를 설정하세요.")

# 출력변수 순서 (테이블 컬럼과 동일한 순서)
FIELDS = [
    "sdate","stime","cjunkook","cjibangDir","cseoulDir","csudj","csudg","csuus",
    "csubs","csugj","csump","csukr","cdjsu","cdgsu","cussu","cbssu","cgjsu",
    "cmpsu","ckrsu","csuyy","cyysu"
]

TABLE = "forecast_traffic"

# 데이터 리스트 찾기
def find_data_list(parsed):
    if parsed is None:
        return []
    if isinstance(parsed, dict):
        if "list" in parsed and isinstance(parsed["list"], (list, dict)):
            if isinstance(parsed["list"], list):
                return parsed["list"]
            elif isinstance(parsed["list"], dict):
                if "item" in parsed["list"]:
                    it = parsed["list"]["item"]
                    return it if isinstance(it, list) else [it]
                return [parsed["list"]]
        if "items" in parsed and isinstance(parsed["items"], dict) and "item" in parsed["items"]:
            it = parsed["items"]["item"]
            return it if isinstance(it, list) else [it]
        if "item" in parsed:
            it = parsed["item"]
            return it if isinstance(it, list) else [it]
        for v in parsed.values():
            if isinstance(v, (dict, list)):
                found = find_data_list(v)
                if found:
                    return found
    elif isinstance(parsed, list):
        return parsed
    return []

# API 호출, type=json 고정 ----------
def fetch_api():
    params = {
        "key": API_KEY,           
        "serviceKey": API_KEY,     
        "type": "json",            
        "numOfRows": "100",
        "pageNo": "1"
    }
    resp = requests.get(API_URL, params=params, timeout=15)
    print("Request URL:", resp.url)
    print("HTTP status:", resp.status_code)
    text = resp.text
    print("Response snippet (first 2000 chars):")
    print(text[:2000])
    if resp.status_code != 200:
        raise RuntimeError(f"HTTP {resp.status_code} returned")
    if text.strip().startswith("{") or text.strip().startswith("["):
        return resp.json()
    else:
        return xmltodict.parse(text)

# 레코드 추출
def extract_tuple(item):
    if not isinstance(item, dict):
        return None
    out = []
    for f in FIELDS:
        val = None
        if f in item:
            val = item[f]
        else:
            for k in item.keys():
                if k.lower() == f.lower():
                    val = item[k]
                    break
        if isinstance(val, dict) and "#text" in val:
            val = val["#text"]
        if val is None:
            val = ""
        out.append(str(val))
    return tuple(out)

#  MySQL 적재
def upsert_rows_mysql(rows):
    if not rows:
        print("No rows to insert.")
        return
    cols = ",".join(FIELDS)
    placeholders = ",".join(["%s"] * len(FIELDS))
    update_cols = [c for c in FIELDS if c not in ("sdate", "stime")]
    update_assign = ",".join([f"{c}=VALUES({c})" for c in update_cols])
    sql = f"INSERT INTO {TABLE} ({cols}) VALUES ({placeholders}) ON DUPLICATE KEY UPDATE {update_assign}, fetched_at = CURRENT_TIMESTAMP"

    try:
        conn = pymysql.connect(
            host=DB_HOST,
            port=DB_PORT,
            user=DB_USER,
            password=DB_PASS,
            db=DB_NAME,
            charset="utf8mb4",
            autocommit=True
        )
    except Exception as e:
        # DB 접속 실패 시 원인 파악에 도움이 되는 메시지 출력
        raise RuntimeError(f"MySQL connection failed: {e}")

    try:
        with conn.cursor() as cur:
            cur.executemany(sql, rows)
        print(f"Upserted {len(rows)} rows into {TABLE}")
    finally:
        conn.close()

# 메인
def main():
    print("Start fetch -> parse -> upsert:", datetime.now().isoformat())
    try:
        parsed = fetch_api()
    except Exception as e:
        print("API fetch error:", e)
        return

    if isinstance(parsed, dict):
        print("Top-level keys:", list(parsed.keys()))
        print("API message/code:", parsed.get("message"), parsed.get("code"))

    data_list = find_data_list(parsed)
    if not data_list:
        print("No items found in parsed response. Check 'list' or 'items' content.")
        return

    print("Found data list length:", len(data_list))
    if len(data_list) > 0 and isinstance(data_list[0], dict):
        print("First item sample keys:", list(data_list[0].keys()))

    rows = []
    for it in data_list:
        t = extract_tuple(it)
        if t:
            rows.append(t)

    if not rows:
        print("No valid rows extracted.")
        return

    try:
        upsert_rows_mysql(rows)
    except Exception as e:
        print("DB upsert error:", e)
        return

    print("Finished:", datetime.now().isoformat())

if __name__ == "__main__":
    main()

Start fetch -> parse -> upsert: 2026-03-31T11:32:43.873312
Request URL: https://data.ex.co.kr/openapi/safeDriving/forecast?key=0007971950&serviceKey=0007971950&type=json&numOfRows=100&pageNo=1
HTTP status: 200
Response snippet (first 2000 chars):
{"count":1,"list":[{"sdate":"20260331","stime":"1200","cjunkook":"5080000","cjibangDir":"410000","cseoulDir":"410000","csudj":"1:38","csudg":"3:30","csuus":"4:10","csubs":"4:30","csugj":"3:20","csump":"3:52","csukr":"2:40","cdjsu":"1:49","cdgsu":"3:30","cussu":"4:10","cbssu":"4:30","cgjsu":"3:20","cmpsu":"4:03","ckrsu":"2:40","csuyy":"1:50","cyysu":"1:50","csudj_bus":"1:30","csudg_bus":"3:30","csuus_bus":"4:10","csubs_bus":"4:30","csugj_bus":"3:20","csump_bus":"0:00","csukr_bus":"2:40","csuyy_bus":"0:00","cdjsu_bus":"1:30","cdgsu_bus":"3:30","cussu_bus":"4:10","cbssu_bus":"4:30","cgjsu_bus":"3:20","cmpsu_bus":"0:00","ckrsu_bus":"2:40","cyysu_bus":"0:00"}],"message":"인증키가 유효합니다.","code":"SUCCESS"}
Top-level keys: ['count', 'list', 'message', 'c